# Домашнее задание 7

Сегодня будем решать задачу _машинного перевода_ с помощью RNN.

1. Построим RNN, обучим на текстах.
2. Построим bi-directional RNN, обучим, сравним качество.

Стоит отметить, что RNN - это не самый популярный и надежный метод из-за проблем с затуханием и взрывом градиентов, а также ограниченной способности захватывать долгосрочные зависимости в тексте.
Более улучшенные и эффективные модели перевода (такие как LSTM, GRU и трансформеры) вы узнаете в блоке NLP.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

In [2]:
# test pad_sequence 
a = torch.ones(1, 5)
b = torch.ones(2, 5)
c = torch.ones(4, 5)
pad_seq = pad_sequence([a, b, c], padding_value=0)
pad_seq, pad_seq.shape

(tensor([[[1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.]],
 
         [[0., 0., 0., 0., 0.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.]],
 
         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [1., 1., 1., 1., 1.]],
 
         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [1., 1., 1., 1., 1.]]]),
 torch.Size([4, 3, 5]))

In [3]:
# same with batch_first=True
a = torch.ones(1, 5)
b = torch.ones(2, 5)
c = torch.ones(4, 5)
pad_seq = pad_sequence([a, b, c], padding_value=0, batch_first=True)
pad_seq, pad_seq.shape

(tensor([[[1., 1., 1., 1., 1.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],
 
         [[1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1.]]]),
 torch.Size([3, 4, 5]))

In [4]:
# Загружаем датасет

In [5]:
def load_pairs(file_path):
    pairs = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            pair = line.strip().split("\t")
            pairs.append(pair)
    return pairs


pairs = load_pairs("/kaggle/input/datasets/andreykurdyubov/hw7-pairs/hw7-pairs.txt")

In [6]:
pairs[200:205]

[['Back off.', 'Посторонитесь.'],
 ['Be a man.', 'Будь мужчиной!'],
 ['Be brave.', 'Будь храбр.'],
 ['Be brief.', 'Будь краток.'],
 ['Be quiet.', 'Тихо.']]

In [7]:
# Делаем нужные предобработки для задачи перевода

In [8]:
# Определение специальных токенов
PAD_TOKEN = "<PAD>"
EOS_TOKEN = "<EOS>"
SOS_TOKEN = "<SOS>"
UNK_TOKEN = "<UNK>"


# Функция токенизации предложения: приводит все символы к нижнему регистру и разбивает предложение на слова
def tokenize(sentence):
    return sentence.lower().split()


# Функция для построения словарей для английских и русских слов на основе пар предложений
def build_vocab(pairs):
    eng_vocab = set()
    rus_vocab = set()
    for eng_sentence, rus_sentence in pairs:
        eng_vocab.update(tokenize(eng_sentence))
        rus_vocab.update(tokenize(rus_sentence))
    return eng_vocab, rus_vocab


# Функция для создания отображений слово -> индекс и индекс -> слово
def create_mappings(vocab):
    vocab = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN] + sorted(vocab)
    word2int = {word: i for i, word in enumerate(vocab)}
    int2word = {i: word for word, i in word2int.items()}
    return word2int, int2word


# Создание словарей для английских и русских предложений на основе пар
english_vocab, russian_vocab = build_vocab(pairs)

# Создание отображений с добавлением специальных токенов
eng_word2int, eng_int2word = create_mappings(english_vocab)
rus_word2int, rus_int2word = create_mappings(russian_vocab)

# Печать размеров словарей (с учетом 4 специальных токенов)
print("English vocabulary size:", len(english_vocab) + 4)
print("Russian vocabulary size:", len(russian_vocab) + 4)

# Пример использования: кодирование английского и русского предложения
eng_example = "Who are you"
rus_example = "как ты"

# Кодирование с учетом UNK_TOKEN для неизвестных слов
eng_encoded = np.array(
    [eng_word2int.get(word, eng_word2int[UNK_TOKEN]) for word in tokenize(eng_example)],
    dtype=np.int32,
)
rus_encoded = np.array(
    [rus_word2int.get(word, rus_word2int[UNK_TOKEN]) for word in tokenize(rus_example)],
    dtype=np.int32,
)

print("English text encoded:", eng_encoded)
print("Russian text encoded:", rus_encoded)

# Декодирование: восстановление текста из кодов
decoded_eng = " ".join([eng_int2word[i] for i in eng_encoded])
decoded_rus = " ".join([rus_int2word[i] for i in rus_encoded])

print("Decoded English:", decoded_eng)
print("Decoded Russian:", decoded_rus)


# Определение класса датасета для перевода
class TranslationDataset(Dataset):
    def __init__(self, pairs, eng_word2int, rus_word2int):
        self.pairs = pairs
        self.eng_word2int = eng_word2int
        self.rus_word2int = rus_word2int

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        # Кодирование английского предложения и добавление EOS токена
        eng_tensor = torch.tensor(
            [
                self.eng_word2int.get(word, self.eng_word2int[UNK_TOKEN])
                for word in tokenize(eng)
            ]
            + [self.eng_word2int[EOS_TOKEN]],
            dtype=torch.long,
        )
        # Кодирование русского предложения и добавление EOS токена
        rus_tensor = torch.tensor(
            [
                self.rus_word2int.get(word, self.rus_word2int[UNK_TOKEN])
                for word in tokenize(rus)
            ]
            + [self.rus_word2int[EOS_TOKEN]],
            dtype=torch.long,
        )
        return eng_tensor, rus_tensor


# Функция для объединения батчей: паддинг (дополнение) предложений до одной длины в батче
def collate_fn(batch):
    eng_batch, rus_batch = zip(*batch)
    eng_batch_padded = pad_sequence(
        eng_batch, batch_first=True, padding_value=eng_word2int[PAD_TOKEN]
    )
    rus_batch_padded = pad_sequence(
        rus_batch, batch_first=True, padding_value=rus_word2int[PAD_TOKEN]
    )
    return eng_batch_padded, rus_batch_padded


# Создание экземпляра датасета и загрузчика данных
translation_dataset = TranslationDataset(pairs, eng_word2int, rus_word2int)
batch_size = 64
translation_dataloader = DataLoader(
    translation_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=collate_fn,
)

# Печать информации о количестве образцов и батчей в датасете
print("Translation samples: ", len(translation_dataset))
print("Translation batches: ", len(translation_dataloader))

English vocabulary size: 34195
Russian vocabulary size: 86949
English text encoded: [33425  2292 34085]
Russian text encoded: [25873 77975]
Decoded English: who are you
Decoded Russian: как ты
Translation samples:  323711
Translation batches:  5057


In [9]:
# проверка размерности батчей
c = 0
for x in translation_dataloader:
    print(f"Batch #{c}")
    print(f"eng_batch_padded {x[0].shape}")
    print(f"rus_batch_padded {x[1].shape}")
    c += 1
    if c >= 3:
        break


Batch #0
eng_batch_padded torch.Size([64, 13])
rus_batch_padded torch.Size([64, 13])
Batch #1
eng_batch_padded torch.Size([64, 13])
rus_batch_padded torch.Size([64, 12])
Batch #2
eng_batch_padded torch.Size([64, 24])
rus_batch_padded torch.Size([64, 20])


## Простая RNN

Начнем свои эксперименты с простой `RNN` - без bidirectional и с одним слоем.

### Задание №1

Добавьте недостающие части в класс `Encoder` и сдайте в ЛМС код класса.

In [10]:
# torch.flip?
x = torch.arange(8).view(2, 2, 2)
y = torch.flip(x, [0, 2])
x, y

(tensor([[[0, 1],
          [2, 3]],
 
         [[4, 5],
          [6, 7]]]),
 tensor([[[5, 4],
          [7, 6]],
 
         [[1, 0],
          [3, 2]]]))

In [11]:
# nn.RNN?
ref_rnn = nn.RNN(10, 32)
print(ref_rnn.weight_hh_l0.shape)
print(ref_rnn.weight_ih_l0.shape)
print(ref_rnn.bias_hh_l0.shape)
print(ref_rnn.bias_ih_l0.shape)

torch.Size([32, 32])
torch.Size([32, 10])
torch.Size([32])
torch.Size([32])


In [12]:
x = torch.randn(1, 10)
x

tensor([[ 2.1029,  2.1465,  0.2791, -0.0386,  0.0225,  1.3392,  1.7362, -1.1855,
         -1.4161, -0.7778]])

In [13]:
input_size = 10
hidden_size = 5
rnn = nn.RNN(input_size, hidden_size, batch_first=True)

x = torch.randn(1, 4, 10)
rnn(x)

(tensor([[[ 0.3965,  0.4136,  0.4623,  0.6570, -0.0532],
          [-0.6555,  0.3758, -0.0203, -0.5082, -0.4854],
          [-0.6763, -0.6518,  0.4646,  0.5220, -0.0754],
          [-0.8479, -0.1969,  0.9183, -0.3108,  0.5630]]],
        grad_fn=<TransposeBackward1>),
 tensor([[[-0.8479, -0.1969,  0.9183, -0.3108,  0.5630]]],
        grad_fn=<StackBackward0>))

In [14]:
class Encoder(nn.Module):
    def __init__(
        self, vocab_size: int, embed_size: int, hidden_size: int, num_layers: int = 1
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # Добавьте слой RNN
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        x = torch.flip(x, [1])
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

In [15]:
x = torch.ones(2, 3, dtype=torch.int64)
enc = Encoder(vocab_size=10, embed_size=5, hidden_size=6, num_layers=4)
enc(x)

(tensor([[[ 0.1124, -0.2246,  0.2138, -0.5054, -0.4683, -0.2582],
          [-0.3038, -0.4778, -0.0824, -0.5963, -0.4557, -0.0830],
          [-0.4862, -0.1932,  0.1167, -0.6506, -0.1719,  0.1387]],
 
         [[ 0.1124, -0.2246,  0.2138, -0.5054, -0.4683, -0.2582],
          [-0.3038, -0.4778, -0.0824, -0.5963, -0.4557, -0.0830],
          [-0.4862, -0.1932,  0.1167, -0.6506, -0.1719,  0.1387]]],
        grad_fn=<TransposeBackward1>),
 tensor([[[-0.7095,  0.4717, -0.7405, -0.4256, -0.6447, -0.3699],
          [-0.7095,  0.4717, -0.7405, -0.4256, -0.6447, -0.3699]],
 
         [[ 0.2203, -0.3471, -0.6555, -0.2337,  0.1828, -0.5523],
          [ 0.2203, -0.3471, -0.6555, -0.2337,  0.1828, -0.5523]],
 
         [[-0.8140, -0.1252, -0.6427,  0.7212, -0.3875,  0.5205],
          [-0.8140, -0.1252, -0.6427,  0.7212, -0.3875,  0.5205]],
 
         [[-0.4862, -0.1932,  0.1167, -0.6506, -0.1719,  0.1387],
          [-0.4862, -0.1932,  0.1167, -0.6506, -0.1719,  0.1387]]],
        grad_fn=<Stac

### Задание №2

Добавьте недостающий код в `Decoder` и сдайте в ЛМС код класса.

In [16]:
class Decoder(nn.Module):
    def __init__(
        self, vocab_size: int, embed_size: int, hidden_size: int, num_layers: int = 1
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # Добавьте слой RNN и линейный слой, который преобразует выход RNN в размер словаря
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)    

    def forward(self, x: torch.Tensor, hidden: torch.Tensor | None):
        out = self.embedding(x)
        out, hidden = self.rnn(out, hidden)
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden

In [17]:
x = torch.ones(2, 3, dtype=torch.int64)
enc = Encoder(vocab_size=10, embed_size=5, hidden_size=6, num_layers=4)
output, hidden = enc(x)
dec = Decoder(vocab_size=10, embed_size=1, hidden_size=6, num_layers=4)
out, hidden = dec(x, hidden)
out, hidden

(tensor([[ 0.1286, -0.4815,  0.0340,  0.0609,  0.6776,  0.2531, -0.2406,  0.1493,
           0.0399,  0.1388,  0.1018, -0.2693,  0.1945,  0.2797,  0.5022,  0.4889,
          -0.3086,  0.0468, -0.3115,  0.1303,  0.2141, -0.4402,  0.0868,  0.1802,
           0.6343,  0.4439, -0.2556,  0.1559, -0.1421,  0.1626],
         [ 0.1286, -0.4815,  0.0340,  0.0609,  0.6776,  0.2531, -0.2406,  0.1493,
           0.0399,  0.1388,  0.1018, -0.2693,  0.1945,  0.2797,  0.5022,  0.4889,
          -0.3086,  0.0468, -0.3115,  0.1303,  0.2141, -0.4402,  0.0868,  0.1802,
           0.6343,  0.4439, -0.2556,  0.1559, -0.1421,  0.1626]],
        grad_fn=<ViewBackward0>),
 tensor([[[ 0.6333, -0.3196, -0.3336,  0.4358, -0.2402,  0.2732],
          [ 0.6333, -0.3196, -0.3336,  0.4358, -0.2402,  0.2732]],
 
         [[ 0.3991, -0.1741, -0.5947,  0.4777, -0.1291, -0.5766],
          [ 0.3991, -0.1741, -0.5947,  0.4777, -0.1291, -0.5766]],
 
         [[-0.4102,  0.6124,  0.7594,  0.0191, -0.3510,  0.0732],
       

In [18]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda', index=0)

In [19]:
eng_vocab_size = len(eng_word2int)
rus_vocab_size = len(rus_word2int)
embed_size = 256
hidden_size = 512
num_layers = 1

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
encoder = Encoder(eng_vocab_size, embed_size, hidden_size, num_layers).to(DEVICE)
decoder = Decoder(rus_vocab_size, embed_size, hidden_size, num_layers).to(DEVICE)

In [21]:
input_tensor = torch.tensor([[1, 2, 3], [1, 2, 3], [1, 2, 3]])
input_tensor.view(9, -1)

tensor([[1],
        [2],
        [3],
        [1],
        [2],
        [3],
        [1],
        [2],
        [3]])

In [20]:
def translate(encoder, decoder, sentence, eng_word2int, rus_int2word, max_length=15):
    # Переводим модели в режим оценки (inference)
    encoder.eval()
    decoder.eval()

    # Отключаем вычисление градиентов для ускорения и уменьшения использования памяти
    with torch.inference_mode():
        # Преобразуем входное предложение в тензор и добавляем EOS токен в конце
        input_tensor = torch.tensor(
            [eng_word2int[word] for word in tokenize(sentence)]
            + [eng_word2int[EOS_TOKEN]],
            dtype=torch.long,
        )
        input_tensor = input_tensor.view(1, -1).to(DEVICE)  # batch_first=True

        # Пропускаем входное предложение через энкодер
        _, encoder_hidden = encoder(input_tensor)
        # Инициализируем скрытое состояние декодера скрытым состоянием энкодера
        decoder_hidden = encoder_hidden

        decoded_words = []
        last_word = torch.tensor([[eng_word2int[SOS_TOKEN]]]).to(DEVICE)
        for _ in range(max_length):
            # Пропускаем последний предсказанный токен через декодер
            logits, decoder_hidden = decoder(last_word, decoder_hidden)
            # Жадный перебор: выбираем токен с максимальной вероятностью - можно было и с температурой, попробуйте в качестве эксперименте
            next_token = logits.argmax(dim=1)
            last_word = next_token.unsqueeze(0).to(DEVICE)
            if next_token.item() == rus_word2int[EOS_TOKEN]:
                break
            else:
                decoded_words.append(rus_int2word.get(next_token.item()))

    # Возвращаем переведенные слова как строку
    return " ".join(decoded_words)

In [21]:
sentence = "just do it"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: еноты. сформулировать прабабушек? бесплатно засохнет. опухшие целью подавала сочувствии приезжаешь. было? пища? столб. мвф уголках


Пока перевод получается странный, но мы ведь пока ничего не обучали.

In [22]:
import random
import tqdm
import torch.nn as nn

# Функция потерь (исключая паддинг)
loss_fn = nn.CrossEntropyLoss(ignore_index=eng_word2int[PAD_TOKEN])

# Оптимизаторы
encoder_optimizer = optim.AdamW(encoder.parameters())
decoder_optimizer = optim.AdamW(decoder.parameters())

# Количество эпох
num_epochs = 1

# Тренировочный цикл
encoder.train()
decoder.train()

for epoch in range(num_epochs):
    for i, (input_tensor, target_tensor) in tqdm.tqdm(enumerate(translation_dataloader)):
        input_tensor, target_tensor = input_tensor.to(DEVICE), target_tensor.to(DEVICE)

        # Обнуление градиентов обоих оптимизаторов
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        target_length = target_tensor.size(1)

        # Энкодер
        _, encoder_hidden = encoder(input_tensor)

        # Декодер
        decoder_input = torch.full(
            (batch_size, 1), eng_word2int[SOS_TOKEN], dtype=torch.long
        ).to(DEVICE)
        decoder_hidden = encoder_hidden

        # Случайный выбор индекса слова из целевой последовательности
        random_word_index = random.randint(0, target_length - 1)

        loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)
        for di in range(target_length):
            logits, _ = decoder(decoder_input, decoder_hidden)

            # Вычисление потерь только для случайно выбранного слова
            loss = loss + loss_fn(logits, target_tensor[:, di])

            decoder_input = target_tensor[:, di].reshape(
                batch_size, 1
            )  # Teacher forcing (принудительное обучение)

        # Обратное распространение ошибки и шаг оптимизации
        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()

        if i % 100 == 0:
            # Печать потерь каждые 100 батчей
            print(f"Epoch {epoch}, Batch {i}, Loss: {loss.item() / target_length:.4f}")

3it [00:00,  6.06it/s]

Epoch 0, Batch 0, Loss: 11.3994


103it [00:12,  9.23it/s]

Epoch 0, Batch 100, Loss: 6.5947


203it [00:24,  9.81it/s]

...

4903it [09:56,  8.81it/s]

Epoch 0, Batch 4900, Loss: 4.6831


5003it [10:09,  8.13it/s]

Epoch 0, Batch 5000, Loss: 6.8021


5057it [10:15,  8.21it/s]


### Задание №3
Попробуйте перевести предложение "Where is Tom?".
Сдайте в ЛМС перевод.

In [23]:
sentence = "Where is Tom?"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: где говорят по-французски.


In [26]:
sentence = "Sex"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: не хватает денег.


## Bidirectional RNN

Теперь попробуем использовать двунаправленную RNN (bidirectional RNN) в энкодере,
что позволяет модели учитывать информацию из обеих сторон последовательности — как слева направо, так и справа налево.

Декодер остается односторонним.

### Задание №4

Допишите недостающий код в `Encoder` и сдайте на в ЛМС код класса.

In [27]:
class Encoder(nn.Module):
    def __init__(
        self, vocab_size: int, embed_size: int, hidden_size: int, num_layers: int = 1
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # Добавьте двунаправленную RNN
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True, bidirectional=True)

    def forward(self, x: torch.Tensor):
        embedded = self.embedding(x)
        outputs, hidden = self.rnn(embedded)
        # Двунаправленная RNN возвращает два скрытых состояния: одно для каждого направления.
        # Объединяем их в одно скрытое состояние.
        hidden = torch.cat((hidden[0, :, :], hidden[1, :, :]), dim=1).unsqueeze(0)
        return outputs, hidden

In [28]:
x = torch.ones(2, 3, dtype=torch.int64)
enc = Encoder(vocab_size=10, embed_size=5, hidden_size=6, num_layers=4)
enc(x)

(tensor([[[ 0.0610,  0.3994, -0.5545,  0.6419, -0.1855,  0.3154,  0.1411,
            0.2194, -0.1210, -0.2987,  0.0775, -0.1848],
          [-0.1555,  0.4206, -0.8269,  0.3972, -0.4545,  0.6175,  0.3281,
            0.3449, -0.0162, -0.2153,  0.1540, -0.0579],
          [-0.3139,  0.3667, -0.8269,  0.3050, -0.4586,  0.5923,  0.2287,
            0.4388, -0.0879, -0.1185,  0.0200, -0.1889]],
 
         [[ 0.0610,  0.3994, -0.5545,  0.6419, -0.1855,  0.3154,  0.1411,
            0.2194, -0.1210, -0.2987,  0.0775, -0.1848],
          [-0.1555,  0.4206, -0.8269,  0.3972, -0.4545,  0.6175,  0.3281,
            0.3449, -0.0162, -0.2153,  0.1540, -0.0579],
          [-0.3139,  0.3667, -0.8269,  0.3050, -0.4586,  0.5923,  0.2287,
            0.4388, -0.0879, -0.1185,  0.0200, -0.1889]]],
        grad_fn=<TransposeBackward1>),
 tensor([[[ 0.1226,  0.7078, -0.8822, -0.4576,  0.5288, -0.3237, -0.3601,
            0.0944, -0.3164,  0.0307, -0.3628, -0.0363],
          [ 0.1226,  0.7078, -0.8822, -

### Задание №5

Допишите класс `Decoder` и сдайте в ЛМС его реализацию.

In [35]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_size, vocab_size)  

    def forward(self, x, hidden):
        out = self.embedding(x)
        out, hidden = self.rnn(out, hidden)
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden

In [36]:
eng_vocab_size = len(eng_word2int)
ita_vocab_size = len(rus_word2int)
embed_size = 256
hidden_size = 512
num_layers = 1

encoder = Encoder(eng_vocab_size, embed_size, hidden_size, num_layers).to(DEVICE)
decoder = Decoder(ita_vocab_size, embed_size, hidden_size * 2, num_layers).to(DEVICE)

In [37]:
sentence = "just do it"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: разговорчивым. экватор сума чувствительная умрёте. молча, понижения 9,4% разожгли переполнена. ножницы? неразговорчив. мечтаниям трафик. деяния.


In [38]:
import random
import tqdm
import torch.nn as nn
import torch.optim as optim

loss_fn = nn.CrossEntropyLoss(ignore_index=eng_word2int[PAD_TOKEN])

encoder_optimizer = optim.AdamW(encoder.parameters())
decoder_optimizer = optim.AdamW(decoder.parameters())

num_epochs = 1

encoder.train()
decoder.train()

for epoch in range(num_epochs):
    for i, (input_tensor, target_tensor) in tqdm.tqdm(enumerate(translation_dataloader)):
        input_tensor, target_tensor = input_tensor.to(DEVICE), target_tensor.to(DEVICE)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        target_length = target_tensor.size(1)

        _, encoder_hidden = encoder(input_tensor)

        decoder_input = torch.full(
            (batch_size, 1), eng_word2int[SOS_TOKEN], dtype=torch.long
        ).to(DEVICE)
        decoder_hidden = encoder_hidden

        random_word_index = random.randint(0, target_length - 1)

        loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)
        for di in range(target_length):
            logits, decoder_hidden = decoder(decoder_input, decoder_hidden)

            loss = loss + loss_fn(logits, target_tensor[:, di])

            decoder_input = target_tensor[:, di].reshape(batch_size, 1)

        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()

        if i % 100 == 0:
            print(f"Epoch {epoch}, Batch {i}, Loss: {loss.item() / target_length:.4f}")

3it [00:00,  8.04it/s]

Epoch 0, Batch 0, Loss: 11.4113


103it [00:17,  6.07it/s]

Epoch 0, Batch 100, Loss: 6.3986


203it [00:33,  6.34it/s]

...

4901it [13:29,  4.73it/s]

Epoch 0, Batch 4900, Loss: 5.8330


5003it [13:46,  6.47it/s]

Epoch 0, Batch 5000, Loss: 5.5100


5057it [13:56,  6.05it/s]


In [39]:
sentence = "just do it"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: это было мэри.


In [40]:
sentence = "Sex"
translated_sentence = translate(encoder, decoder, sentence, eng_word2int, rus_int2word)
print("Translated:", translated_sentence)

Translated: мы с томом разговариваю.


Выбить хорошее качество обучения, используя только лишь RNN и небольшой датасет, сложно.

Не забывайте, что в семинаре у нас было ~400 Мб текстов одного языка, а здесь всего лишь 28 Мб и на двух языках.
Можем сделать вывод, что для обучения хорошей модели нужно много текстовых данных.

Помимо этого, для серьезного обучения стоит использовать более продвинутые сети: те же GRU и LSTM покажут себя лучше.
А еще лучше будет работать трансформер, о котором вы узнаете в следующем уроке.